# Исследование данных и проектирование решения

Цель ноутбука — проверить свойства данных, обосновать выбранный пайплайн и зафиксировать результаты экспериментов по генерации кандидатов.

## 1. Постановка задачи

### 1.1. Цель

Для каждого поискового запроса необходимо вернуть до 50 наиболее релевантных объявлений.

### 1.2. Метрика качества

Основная метрика — Recall@50, усреднённый по поисковым запросам.

### 1.3. Ограничения

Решение должно быть воспроизводимым локально и не использовать внешние API во время инференса.

## 2. Предполагаемый пайплайн решения

Поисковый запрос проходит нормализацию, географическое ограничение, несколько retrieval-веток, объединение кандидатов и итоговое ранжирование до top-50.

## 3. Загрузка и первичный обзор данных

Сначала фиксируем пути, лениво открываем Parquet-файлы и ограничиваем размер отображаемых таблиц.

In [4]:
import re
from pathlib import Path

import polars as pl

pl.Config.set_tbl_rows(8)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(70)

project_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "dataset").exists()
)
data_dir = project_root / "dataset"

train_path = data_dir / "train.parquet"
queries_path = data_dir / "benchmark_queries.parquet"
items_path = data_dir / "benchmark_items.parquet"

for path in (train_path, queries_path, items_path):
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

train = pl.scan_parquet(train_path)
benchmark_queries = pl.scan_parquet(queries_path)
benchmark_items = pl.scan_parquet(items_path)

tables = {
    "train": train,
    "benchmark_queries": benchmark_queries,
    "benchmark_items": benchmark_items,
}

query_context_columns = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]


def normalized_query(column: str = "search_query") -> pl.Expr:
    return (
        pl.col(column)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


def row_count(table: pl.LazyFrame) -> int:
    return table.select(pl.len()).collect().item()

### 3.1. Размеры и схемы таблиц

Проверяем число строк, столбцов и типы признаков во всех трёх таблицах.

In [5]:
table_summary = pl.DataFrame([
    {
        "table": name,
        "rows": row_count(table),
        "columns": len(table.collect_schema()),
    }
    for name, table in tables.items()
])

schema_summary = pl.DataFrame([
    {
        "table": table_name,
        "column": column_name,
        "dtype": str(dtype),
    }
    for table_name, table in tables.items()
    for column_name, dtype in table.collect_schema().items()
])

display(table_summary)
display(schema_summary)

table,rows,columns
str,i64,i64
"""train""",497673,19
"""benchmark_queries""",2452,6
"""benchmark_items""",189212,14


table,column,dtype
str,str,str
"""train""","""search_query""","""String"""
"""train""","""search_location_id""","""Int64"""
"""train""","""search_is_delivery_search""","""Int32"""
"""train""","""search_infm_params_text""","""String"""
…,…,…
"""benchmark_items""","""item_infm_params_text""","""String"""
"""benchmark_items""","""item_id""","""String"""
"""benchmark_items""","""item_description_raw""","""String"""
"""benchmark_items""","""item_category_id""","""Int64"""


### 3.2. Примеры обучающих наблюдений

Просматриваем небольшой срез связей между поисковым контекстом и объявлениями.

In [6]:
train.head(5).collect()

search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
str,i64,i32,str,i64,str,f64,f64,"decimal[27,15]",i64,"decimal[18,15]",i64,"decimal[17,15]",bool,bool,str,str,str,i64
"""скупка телевизоров""",652430,0,"""""",114,"""Скупка б/у техники""",91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,true,false,"""Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимости за…","""e8b685dffe1a408e""","""Скупаю практически любую современную новую и б/у технику(обязательно р…",114
"""автоподбор""",640860,0,"""Рейтинг пользователя 4 звезды и выше""",114,"""Автоподбор Разовый осмотр автомобиля""",462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,false,false,"""Вид услуги Место оказания услуг Нижний Новгород, Советский район, жило…","""92e1b0446f827b59""","""🚗 Автоподбор и выездная диагностика автомобиля в Нижнем Новгороде и об…",114
"""баня на дровах""",653240,0,"""Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоров…",114,"""Баня на дровах ""Прованс"" на Цветочной""",null,null,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,false,false,"""Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург, сад…","""624846856ce81d69""","""В ритме современной жизни так сложно найти момент, чтобы остановиться…",114
"""изготовление госномера на авто""",634670,0,"""Вид услуги Оборудование, производство""",114,"""Изготовление дубликатов авто номеров, гос номеров""",14.0,4.714286,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,false,false,"""Вид услуги Оборудование, производство Тип услуги Производство, обработ…","""45b8628b9c6e7b85""","""Изготовим дубликат номера по утере или износу на официальных основания…",114
"""укладка плитки""",658430,0,"""Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка""",114,"""Ремонт и отделка квартир под ключ""",1.0,5.0,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,false,false,"""Вид услуги Ремонт и отделка Тип услуги Ремонт квартир и домов под ключ…","""c95a4a7daf2a967f""","""отделочные работы любой сложности.""",114


### 3.3. Примеры benchmark-запросов и объявлений

Отдельно просматриваем входные запросы и корпус объявлений, среди которых нужно искать кандидатов.

In [7]:
display(benchmark_queries.head(5).collect())
display(benchmark_items.head(5).collect())

query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
str,str,i64,i32,str,i64
"""70DfDUpwjxB4lzFd""","""перевозки владикавказ тбилиси""",649820,0,"""""",114
"""JTrdTaZJvSiLPkXj""","""обзвон по базе""",107620,0,"""Вид услуги Деловые услуги""",114
"""LZCZNoVG4AFUkVRJ""","""липоредукция подбородка""",637640,0,"""Вид услуги Красота, здоровье""",114
"""660ac9QVtXkRxZC3""","""подъемник 4 х стоечный""",662810,0,"""""",114
"""YgHcM9MVbxKnxD1e""","""монтаж видеодомофонов""",642790,0,"""""",114


item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
str,f64,f64,"decimal[27,15]",i64,"decimal[18,15]",i64,"decimal[17,15]",bool,bool,str,str,str,i64
"""Ремонт/выкуп компьют. и ноутбуков с выездом на дом""",57.0,5.0,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,false,false,"""Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип ст…","""111eb8b979577d79""","""Выезд на дом в любое время, вплоть до 23:00 ● Диагностика устройств; …",114
"""Обучение ребенка чтению""",7.0,5.0,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,false,false,"""Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 …","""75fc8e10f5a66fc4""","""Дорогие родители и маленькие книголюбы! Я, детский психолог/нейропси…",114
"""Афрокудри 5+""",14.0,5.0,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,true,false,"""Вид услуги Красота, здоровье Место оказания услуг городской округ Иван…","""3f6ae81704565b9c""","""ВНИМАНИЕ Укладка !!!! 🥳 АФРОкудри отличный вариант замены хим завивки…",114
"""Монтаж малых архитектурных форм""",2.0,5.0,5000.000000000000000,2058739,37.576183000000000,637640,55.662734999999998,false,false,"""Вид услуги Строительство Место оказания услуг Севастопольский пр-т, 28…","""0618dec37a59a21a""","""Оказываем все виды услуг по сборке и монтажу игровых площадок , а так …",114
"""Репетитор по истории 10 класс ЕГЭ""",19.0,5.0,1500.000000000000000,86456,44.750473999999997,624850,48.786008000000002,false,false,"""Вид услуги Обучение, курсы Место оказания услуг Волгоградская обл., Во…","""13da2c81574677ed""","""🚀 Сдай ЕГЭ по истории на 80+ за 6 месяцев! Мечтаешь поступить в вуз с…",114


## 4. Проверка качества данных

Цель раздела — установить реальную гранулярность данных и обнаружить проблемы, влияющие на обучение и валидацию.

### 4.1. Единица наблюдения

Сравниваем исходные строки, полностью уникальные строки и уникальные положительные связи «поисковый контекст — объявление».

In [8]:
train_rows = row_count(train)
unique_full_rows = row_count(train.unique())
unique_positive_pairs = row_count(
    train.select([*query_context_columns, "item_id"]).unique()
)

grain_summary = pl.DataFrame({
    "grain": [
        "Все строки",
        "Уникальные полные строки",
        "Уникальные связи контекст-item",
    ],
    "rows": [train_rows, unique_full_rows, unique_positive_pairs],
}).with_columns(
    (pl.col("rows") / train_rows * 100)
    .round(3)
    .alias("share_of_train_pct")
)

grain_summary

grain,rows,share_of_train_pct
str,i64,f64
"""Все строки""",497673,100.0
"""Уникальные полные строки""",467054,93.848
"""Уникальные связи контекст-item""",467054,93.848


### 4.2. Пропущенные значения

Для каждого столбца считаем абсолютное число и долю `null` отдельно по каждой таблице.

In [9]:
def null_profile(table_name: str, table: pl.LazyFrame) -> pl.DataFrame:
    rows = row_count(table)
    counts = table.select(pl.all().null_count()).collect().row(0, named=True)

    return pl.DataFrame({
        "table": [table_name] * len(counts),
        "column": list(counts),
        "null_count": list(counts.values()),
    }).with_columns(
        (pl.col("null_count") / rows * 100)
        .round(4)
        .alias("null_pct")
    )


missing_values = pl.concat([
    null_profile(name, table)
    for name, table in tables.items()
]).filter(
    pl.col("null_count") > 0
).sort(
    ["table", "null_pct"],
    descending=[False, True],
)

missing_values

table,column,null_count,null_pct
str,str,i64,f64
"""benchmark_items""","""item_rating""",17631,9.3181
"""benchmark_items""","""item_rating_reviews_count""",11615,6.1386
"""benchmark_items""","""item_description_raw""",35,0.0185
"""benchmark_items""","""item_longitude""",1,0.0005
…,…,…,…
"""train""","""item_rating""",28232,5.6728
"""train""","""item_rating_reviews_count""",19211,3.8602
"""train""","""item_longitude""",4,0.0008
"""train""","""item_latitude""",4,0.0008


### 4.3. Корректность идентификаторов

Проверяем уникальность benchmark-ID и соответствие идентификаторов ожидаемому шестнадцатеричному формату.

In [10]:
id_checks = pl.DataFrame([
    {
        "field": "benchmark_queries.query_id",
        "rows": row_count(benchmark_queries),
        "unique": benchmark_queries.select(pl.col("query_id").n_unique()).collect().item(),
        "invalid_format": benchmark_queries.select(
            (~pl.col("query_id").str.contains(r"^[0-9a-f]{16}$")).sum()
        ).collect().item(),
    },
    {
        "field": "benchmark_items.item_id",
        "rows": row_count(benchmark_items),
        "unique": benchmark_items.select(pl.col("item_id").n_unique()).collect().item(),
        "invalid_format": benchmark_items.select(
            (~pl.col("item_id").str.contains(r"^[0-9a-f]{16}$")).sum()
        ).collect().item(),
    },
    {
        "field": "train.item_id",
        "rows": row_count(train),
        "unique": train.select(pl.col("item_id").n_unique()).collect().item(),
        "invalid_format": train.select(
            (~pl.col("item_id").str.contains(r"^[0-9a-f]{16}$")).sum()
        ).collect().item(),
    },
])

id_checks

field,rows,unique,invalid_format
str,i64,i64,i64
"""benchmark_queries.query_id""",2452,2452,2452
"""benchmark_items.item_id""",189212,189212,0
"""train.item_id""",497673,344825,0


### 4.4. Полные и смысловые дубликаты

Точный повтор строки и повтор положительной связи считаются отдельно: это разные виды кратности.

In [11]:
relation_counts = (
    train
    .group_by([*query_context_columns, "item_id"])
    .len()
)

duplicate_summary = pl.DataFrame({
    "duplicate_type": [
        "Полные строки сверх первой копии",
        "Связи контекст-item сверх первой копии",
    ],
    "excess_rows": [
        train_rows - unique_full_rows,
        relation_counts.select((pl.col("len") - 1).sum()).collect().item(),
    ],
    "max_multiplicity": [
        train.group_by(list(train.collect_schema())).len()
        .select(pl.col("len").max()).collect().item(),
        relation_counts.select(pl.col("len").max()).collect().item(),
    ],
})

duplicate_summary

duplicate_type,excess_rows,max_multiplicity
str,i64,i64
"""Полные строки сверх первой копии""",30619,171
"""Связи контекст-item сверх первой копии""",30619,171


### 4.5. Аномальные значения

Считаем специальные и физически невозможные значения, не смешивая их с обычными пропусками.

In [12]:
anomaly_checks = benchmark_items.select(
    (pl.col("item_price") == -1).sum().alias("price_minus_one"),
    (pl.col("item_price") == 0).sum().alias("price_zero"),
    (pl.col("item_price") < -1).sum().alias("price_below_minus_one"),
    (
        pl.col("item_rating").is_not_null()
        & ~pl.col("item_rating").is_between(0, 5)
    ).sum().alias("rating_outside_0_5"),
    (
        pl.col("item_latitude").is_not_null()
        & ~pl.col("item_latitude").is_between(-90, 90)
    ).sum().alias("invalid_latitude"),
    (
        pl.col("item_longitude").is_not_null()
        & ~pl.col("item_longitude").is_between(-180, 180)
    ).sum().alias("invalid_longitude"),
).collect()

anomaly_checks

price_minus_one,price_zero,price_below_minus_one,rating_outside_0_5,invalid_latitude,invalid_longitude
u32,u32,u32,u32,u32,u32
15248,1340,0,0,0,0


## 5. Анализ поисковых запросов

Проверяем разнообразие, длину, географию и структурные признаки запросов.

### 5.1. Уникальность и повторяемость запросов

Сравниваем исходные строки запросов с простой нормализацией регистра и пробелов.

In [13]:
normalized_train = train.with_columns(
    normalized_query().alias("query_norm")
)

query_uniqueness = normalized_train.select(
    pl.len().alias("rows"),
    pl.col("search_query").n_unique().alias("unique_raw_queries"),
    pl.col("query_norm").n_unique().alias("unique_normalized_queries"),
).collect()

query_frequencies = (
    normalized_train
    .group_by("query_norm")
    .len()
    .sort("len", descending=True)
    .head(20)
    .collect()
)

display(query_uniqueness)
display(query_frequencies)

rows,unique_raw_queries,unique_normalized_queries
u32,u32,u32
497673,74529,73903


query_norm,len
str,u32
"""маникюр""",6540
"""массаж""",5702
"""наращивание ресниц""",5397
"""установка кондиционеров""",4576
…,…
"""торты на заказ""",1682
"""тонировка""",1644
"""поклейка обоев""",1611
"""грузчики""",1540


### 5.2. Длина запросов

Описываем длину в символах и словах устойчивыми квантилями, а не только средним значением.

In [14]:
query_lengths = normalized_train.with_columns(
    pl.col("query_norm").str.len_chars().alias("characters"),
    pl.col("query_norm").str.split(" ").list.len().alias("words"),
)

query_length_summary = query_lengths.select(
    pl.col("characters").median().alias("characters_median"),
    pl.col("characters").quantile(0.90).alias("characters_p90"),
    pl.col("characters").max().alias("characters_max"),
    pl.col("words").median().alias("words_median"),
    pl.col("words").quantile(0.90).alias("words_p90"),
    pl.col("words").max().alias("words_max"),
).collect()

query_length_summary

characters_median,characters_p90,characters_max,words_median,words_p90,words_max
f64,f64,u32,f64,f64,u32
17.0,28.0,113,2.0,4.0,17


### 5.3. География запросов

Считаем число запросов и число уникальных нормализованных форм для каждой локации.

In [15]:
query_locations = (
    normalized_train
    .group_by("search_location_id")
    .agg(
        pl.len().alias("rows"),
        pl.col("query_norm").n_unique().alias("unique_queries"),
    )
    .sort("rows", descending=True)
    .collect()
)

display(query_locations.head(20))
display(query_locations.select(
    pl.len().alias("location_count"),
    pl.col("rows").median().alias("rows_per_location_median"),
    pl.col("rows").max().alias("rows_per_location_max"),
))

search_location_id,rows,unique_queries
i64,u32,u32
637640,36276,15529
653240,25229,10906
107620,12277,7385
633540,11441,5684
…,…,…
659020,5802,3158
642320,5520,2969
644200,5167,2891
624840,4892,2713


location_count,rows_per_location_median,rows_per_location_max
u32,f64,u32
2782,6.0,36276


### 5.4. Категории и фильтры

Проверяем распределение поисковых категорий и долю запросов с заполненными параметрами.

In [16]:
filter_present = (
    pl.col("search_infm_params_text")
    .fill_null("")
    .str.strip_chars()
    .ne("")
)

query_structure = train.select(
    pl.col("search_category").n_unique().alias("category_count"),
    filter_present.mean().alias("filter_present_share"),
    pl.col("search_is_delivery_search").mean().alias("delivery_share"),
).collect()

search_categories = (
    train.group_by("search_category")
    .len()
    .sort("len", descending=True)
    .collect()
)

display(query_structure)
display(search_categories)

category_count,filter_present_share,delivery_share
u32,f64,f64
4,0.670468,0.000024


search_category,len
i64,u32
114,497635
0,34
81,2
10,2


## 6. Анализ объявлений

Оцениваем полноту текстовых, категориальных, географических и числовых признаков объявлений.

### 6.1. Текстовые поля

Для каждого текстового поля считаем пропуски, пустые строки и распределение длины.

In [17]:
text_columns = [
    "item_title_raw",
    "item_infm_params_text",
    "item_description_raw",
]

text_profile = pl.DataFrame([
    {
        "column": column,
        "null_count": benchmark_items.select(
            pl.col(column).null_count()
        ).collect().item(),
        "empty_count": benchmark_items.select(
            (pl.col(column).fill_null("").str.strip_chars() == "").sum()
        ).collect().item(),
        "length_median": benchmark_items.select(
            pl.col(column).fill_null("").str.len_chars().median()
        ).collect().item(),
        "length_p90": benchmark_items.select(
            pl.col(column).fill_null("").str.len_chars().quantile(0.90)
        ).collect().item(),
    }
    for column in text_columns
])

text_profile

column,null_count,empty_count,length_median,length_p90
str,i64,i64,f64,f64
"""item_title_raw""",0,0,35.0,48.0
"""item_infm_params_text""",0,0,738.0,1864.0
"""item_description_raw""",35,35,1054.0,3127.0


### 6.2. Категории и микрокатегории

Проверяем концентрацию объявлений в категориях и микрокатегориях.

In [18]:
item_categories = (
    benchmark_items
    .group_by("item_category_id")
    .len()
    .sort("len", descending=True)
    .collect()
)

item_microcategories = (
    benchmark_items
    .group_by("item_microcat_id")
    .len()
    .sort("len", descending=True)
    .collect()
)

display(item_categories.head(20))
display(item_microcategories.head(20))

item_category_id,len
i64,u32
114,187336
19,357
112,141
40,122
…,…
21,36
88,34
82,32
42,29


item_microcat_id,len
i64,u32
86456,6355
2301563,4272
86467,4064
86470,3858
…,…
4145,2715
86454,2685
2301617,2547
2096704,2490


### 6.3. География объявлений

Проверяем число локаций, заполненность координат и размеры крупнейших географических групп.

In [19]:
item_geography = benchmark_items.select(
    pl.col("item_location_id").n_unique().alias("location_count"),
    pl.col("item_location_id").null_count().alias("missing_location"),
    pl.col("item_latitude").null_count().alias("missing_latitude"),
    pl.col("item_longitude").null_count().alias("missing_longitude"),
).collect()

largest_item_locations = (
    benchmark_items
    .group_by("item_location_id")
    .len()
    .sort("len", descending=True)
    .head(20)
    .collect()
)

display(item_geography)
display(largest_item_locations)

location_count,missing_location,missing_latitude,missing_longitude
u32,u32,u32,u32
2877,0,1,1


item_location_id,len
i64,u32
637640,19543
653240,12145
633540,4494
650400,3659
…,…
624840,1770
653241,1646
628970,1638
634450,1616


### 6.4. Цена и признаки популярности

Используем квантили, чтобы длинные хвосты не скрывались за средним значением.

In [20]:
numeric_profile = benchmark_items.select(
    pl.col("item_price").median().alias("price_median"),
    pl.col("item_price").quantile(0.90).alias("price_p90"),
    pl.col("item_price").quantile(0.99).alias("price_p99"),
    pl.col("item_rating").median().alias("rating_median"),
    pl.col("item_rating_reviews_count").median().alias("reviews_median"),
    pl.col("item_rating_reviews_count").quantile(0.90).alias("reviews_p90"),
    pl.col("item_is_phone_hidden").mean().alias("phone_hidden_share"),
    pl.col("item_is_message_forbidden").mean().alias("message_forbidden_share"),
).collect()

numeric_profile

price_median,price_p90,price_p99,rating_median,reviews_median,reviews_p90,phone_hidden_share,message_forbidden_share
f64,f64,f64,f64,f64,f64,f64,f64
1000.0,7000.0,150000.0,5.0,18.0,144.0,0.155947,0.0266


## 7. Сравнение train и benchmark

Пересечения и различия распределений определяют, насколько локальные эксперименты будут переноситься на benchmark.

### 7.1. Пересечение запросов

Считаем совпадения как в исходном виде, так и после одинаковой нормализации текста.

In [21]:
train_queries = train.select(
    "search_query",
    normalized_query().alias("query_norm"),
).unique()

benchmark_query_texts = benchmark_queries.select(
    "search_query",
    normalized_query().alias("query_norm"),
).unique()

query_overlap = pl.DataFrame({
    "comparison": ["raw", "normalized"],
    "matched_benchmark_queries": [
        row_count(
            benchmark_query_texts.select("search_query").join(
                train_queries.select("search_query"),
                on="search_query",
                how="inner",
            )
        ),
        row_count(
            benchmark_query_texts.select("query_norm").join(
                train_queries.select("query_norm"),
                on="query_norm",
                how="inner",
            )
        ),
    ],
}).with_columns(
    (pl.col("matched_benchmark_queries") / row_count(benchmark_queries) * 100)
    .round(3)
    .alias("matched_pct")
)

query_overlap

comparison,matched_benchmark_queries,matched_pct
str,i64,f64
"""raw""",907,36.99
"""normalized""",932,38.01


### 7.2. Пересечение объявлений

Проверяем, какая доля объявлений из benchmark-корпуса уже наблюдалась в train.

In [22]:
train_item_ids = train.select("item_id").unique()
benchmark_item_ids = benchmark_items.select("item_id").unique()

seen_benchmark_items = row_count(
    benchmark_item_ids.join(train_item_ids, on="item_id", how="inner")
)
total_benchmark_items = row_count(benchmark_item_ids)

item_overlap = pl.DataFrame({
    "seen_in_train": [seen_benchmark_items],
    "benchmark_items": [total_benchmark_items],
    "seen_pct": [round(seen_benchmark_items / total_benchmark_items * 100, 3)],
})

item_overlap

seen_in_train,benchmark_items,seen_pct
i64,i64,f64
18142,189212,9.588


### 7.3. Сдвиг распределений

Сравниваем одинаково рассчитанные признаки запросов в train и benchmark.

In [23]:
def query_distribution_profile(
    table_name: str,
    table: pl.LazyFrame,
) -> pl.DataFrame:
    prepared = table.with_columns(
        normalized_query().alias("query_norm"),
        normalized_query().str.split(" ").list.len().alias("word_count"),
        filter_present.alias("has_filter"),
    )

    return prepared.select(
        pl.lit(table_name).alias("table"),
        pl.len().alias("rows"),
        pl.col("query_norm").n_unique().alias("unique_queries"),
        pl.col("word_count").median().alias("words_median"),
        pl.col("word_count").quantile(0.90).alias("words_p90"),
        pl.col("search_location_id").n_unique().alias("location_count"),
        pl.col("search_category").n_unique().alias("category_count"),
        pl.col("has_filter").mean().alias("filter_share"),
        pl.col("search_is_delivery_search").mean().alias("delivery_share"),
    ).collect()


distribution_shift = pl.concat([
    query_distribution_profile("train", train),
    query_distribution_profile("benchmark", benchmark_queries),
])

distribution_shift

table,rows,unique_queries,words_median,words_p90,location_count,category_count,filter_share,delivery_share
str,u32,u32,f64,f64,u32,u32,f64,f64
"""train""",497673,73903,2.0,4.0,2782,4,0.670468,0.000024
"""benchmark""",2452,2451,3.0,5.0,273,2,0.369494,0.0


## 8. Построение локальной валидации

Валидация должна исключать пересечение одинаковых нормализованных запросов между обучающей и проверочной частями.

### 8.1. Разделение по группам запросов

Хеш-разбиение детерминировано и сохраняет все строки одного нормализованного запроса в одной части данных.

In [24]:
RANDOM_SEED = 42
VALIDATION_BUCKETS = 5

query_split = (
    normalized_train
    .select("query_norm")
    .unique()
    .with_columns(
        (
            pl.col("query_norm").hash(seed=RANDOM_SEED)
            % VALIDATION_BUCKETS
            == 0
        ).alias("is_validation")
    )
)

split_train = normalized_train.join(
    query_split,
    on="query_norm",
    how="left",
)

train_fold = split_train.filter(~pl.col("is_validation")).drop("is_validation")
validation_fold = split_train.filter(pl.col("is_validation")).drop("is_validation")

split_summary = pl.DataFrame({
    "part": ["train", "validation"],
    "rows": [row_count(train_fold), row_count(validation_fold)],
    "queries": [
        train_fold.select(pl.col("query_norm").n_unique()).collect().item(),
        validation_fold.select(pl.col("query_norm").n_unique()).collect().item(),
    ],
})

split_summary

part,rows,queries
str,i64,i64
"""train""",400185,58929
"""validation""",97488,14974


### 8.2. Proxy-разметка в benchmark-корпусе

Оставляем только положительные связи, для которых объявление присутствует среди доступных benchmark-объявлений.

In [25]:
validation_labels = (
    validation_fold
    .join(benchmark_item_ids, on="item_id", how="inner")
    .select("query_norm", "item_id")
    .unique()
)

proxy_coverage = validation_fold.select(
    pl.len().alias("validation_rows"),
).collect().with_columns(
    pl.lit(row_count(validation_labels)).alias("usable_positive_pairs"),
    pl.lit(
        validation_labels.select(
            pl.col("query_norm").n_unique()
        ).collect().item()
    ).alias("usable_queries"),
)

proxy_coverage

validation_rows,usable_positive_pairs,usable_queries
u32,i32,i32
97488,5400,2444


### 8.3. Расчёт Recall@K

Функция принимает уникальные истинные пары и ранжированные кандидаты, затем усредняет полноту отдельно по запросам.

In [26]:
def recall_at_k(
    labels: pl.DataFrame,
    candidates: pl.DataFrame,
    k: int = 50,
) -> float:
    required_label_columns = {"query_norm", "item_id"}
    required_candidate_columns = {"query_norm", "item_id", "score"}

    if not required_label_columns <= set(labels.columns):
        raise ValueError(f"labels должны содержать {required_label_columns}")
    if not required_candidate_columns <= set(candidates.columns):
        raise ValueError(f"candidates должны содержать {required_candidate_columns}")

    truth = {
        query: set(items)
        for query, items in (
            labels.unique()
            .group_by("query_norm")
            .agg(pl.col("item_id"))
            .iter_rows()
        )
    }

    predictions = {
        query: items
        for query, items in (
            candidates
            .sort(["query_norm", "score"], descending=[False, True])
            .unique(["query_norm", "item_id"], keep="first")
            .group_by("query_norm", maintain_order=True)
            .agg(pl.col("item_id").head(k))
            .iter_rows()
        )
    }

    per_query_recall = [
        len(relevant_items & set(predictions.get(query, [])))
        / len(relevant_items)
        for query, relevant_items in truth.items()
    ]

    return sum(per_query_recall) / len(per_query_recall)

### 8.4. Контрольные сегменты

Заранее фиксируем признаки, по которым затем сравним качество retrieval-моделей.

In [27]:
validation_segments = (
    validation_fold
    .with_columns(
        normalized_query().str.split(" ").list.len().alias("word_count"),
        filter_present.alias("has_filter"),
    )
    .group_by("query_norm")
    .agg(
        pl.col("word_count").first(),
        pl.col("has_filter").first(),
        pl.col("search_location_id").first(),
        pl.col("search_category").first(),
        pl.col("item_id").n_unique().alias("relevant_items"),
    )
    .with_columns(
        pl.when(pl.col("word_count") <= 2)
        .then(pl.lit("короткий"))
        .when(pl.col("word_count") <= 4)
        .then(pl.lit("средний"))
        .otherwise(pl.lit("длинный"))
        .alias("query_length_group")
    )
    .collect()
)

validation_segments.head()

query_norm,word_count,has_filter,search_location_id,search_category,relevant_items,query_length_group
str,u32,bool,i64,i64,u32,str
"""пишу жалобы на всех""",4,true,641780,114,2,"""средний"""
"""ремонт айфон аклаунд""",3,true,640860,114,1,"""средний"""
"""грузоперевозки газель химки""",3,true,639740,114,2,"""средний"""
"""реставрация пластика автомобиля""",3,false,107621,114,1,"""средний"""
"""манипулятор с ндс""",3,false,659020,114,4,"""средний"""


## 9. Диагностика гипотез генерации кандидатов

На этом этапе проверяем, существуют ли в данных сигналы, которые оправдывают будущие retrieval-ветки. Это ещё не сравнение готовых моделей.

### 9.1. Исторические совпадения

Оцениваем, для какой доли benchmark-запросов имеются ранее наблюдавшиеся объявления, всё ещё присутствующие в текущем корпусе.

In [28]:
historical_candidates = (
    normalized_train
    .join(
        benchmark_query_texts.select("query_norm"),
        on="query_norm",
        how="inner",
    )
    .join(benchmark_item_ids, on="item_id", how="inner")
    .select("query_norm", "item_id")
    .unique()
)

historical_coverage = historical_candidates.select(
    pl.col("query_norm").n_unique().alias("covered_queries"),
    pl.len().alias("candidate_pairs"),
).collect().with_columns(
    pl.lit(row_count(benchmark_queries)).alias("benchmark_queries"),
).with_columns(
    (pl.col("covered_queries") / pl.col("benchmark_queries") * 100)
    .round(3)
    .alias("covered_queries_pct")
)

historical_coverage

covered_queries,candidate_pairs,benchmark_queries,covered_queries_pct
u32,u32,i32,f64
362,958,2452,14.763


### 9.2. Географический сигнал

Для каждой поисковой локации оцениваем распределение локаций положительных объявлений и покрытие наиболее частой локацией.

In [29]:
location_transitions = (
    train_fold
    .group_by(["search_location_id", "item_location_id"])
    .len()
    .with_columns(
        (
            pl.col("len")
            / pl.col("len").sum().over("search_location_id")
        ).alias("share"),
        pl.col("len")
        .rank(method="ordinal", descending=True)
        .over("search_location_id")
        .alias("location_rank"),
    )
)

location_signal = location_transitions.group_by(
    "search_location_id"
).agg(
    pl.col("item_location_id").n_unique().alias("target_locations"),
    pl.col("share").filter(pl.col("location_rank") == 1).first().alias("top1_share"),
    pl.col("share").filter(
        pl.col("item_location_id") == pl.col("search_location_id")
    ).sum().alias("exact_location_share"),
).sort(
    "top1_share"
).collect()

location_signal.head(20)

search_location_id,target_locations,top1_share,exact_location_share
i64,u32,f64,f64
637680,155,0.055281,0.0
636370,77,0.093484,0.0
637970,8,0.181818,0.181818
639560,5,0.2,0.2
…,…,…,…
651980,4,0.25,0.25
653323,4,0.25,0.25
646130,4,0.25,0.0
633980,4,0.25,0.25


### 9.3. Полезность текстовых полей

Сравниваем долю положительных пар, где все слова запроса встречаются в заголовке, параметрах или полном тексте объявления.

In [30]:
def contains_all_query_tokens(query: str, document: str | None) -> bool:
    query_tokens = set(re.findall(r"\w+", query.lower()))
    document_tokens = set(re.findall(r"\w+", (document or "").lower()))
    return bool(query_tokens) and query_tokens <= document_tokens


text_overlap_sample = (
    validation_fold
    .filter(pl.col("query_norm").hash(seed=42) % 10 == 0)
    .select(
        "query_norm",
        "item_title_raw",
        "item_infm_params_text",
        "item_description_raw",
    )
    .collect()
    .with_columns(
        pl.struct(["query_norm", "item_title_raw"])
        .map_elements(
            lambda row: contains_all_query_tokens(
                row["query_norm"],
                row["item_title_raw"],
            ),
            return_dtype=pl.Boolean,
        )
        .alias("all_tokens_in_title"),
        pl.struct([
            "query_norm",
            "item_title_raw",
            "item_infm_params_text",
        ])
        .map_elements(
            lambda row: contains_all_query_tokens(
                row["query_norm"],
                f'{row["item_title_raw"] or ""} {row["item_infm_params_text"] or ""}',
            ),
            return_dtype=pl.Boolean,
        )
        .alias("all_tokens_in_title_and_params"),
        pl.struct([
            "query_norm",
            "item_title_raw",
            "item_infm_params_text",
            "item_description_raw",
        ])
        .map_elements(
            lambda row: contains_all_query_tokens(
                row["query_norm"],
                " ".join([
                    row["item_title_raw"] or "",
                    row["item_infm_params_text"] or "",
                    row["item_description_raw"] or "",
                ]),
            ),
            return_dtype=pl.Boolean,
        )
        .alias("all_tokens_in_all_text"),
    )
)

text_overlap_sample.select(
    pl.len().alias("sample_rows"),
    pl.col("all_tokens_in_title").mean(),
    pl.col("all_tokens_in_title_and_params").mean(),
    pl.col("all_tokens_in_all_text").mean(),
)

sample_rows,all_tokens_in_title,all_tokens_in_title_and_params,all_tokens_in_all_text
u32,f64,f64,f64
49515,0.360436,0.469575,0.677956


### 9.4. Семантический поиск

Гипотеза проверяется отдельным retrieval-экспериментом на зафиксированной validation-выборке. До выбора embedding-модели здесь не создаём псевдопроверку, которая не измеряет семантический поиск.

### 9.5. Связь фильтров и микрокатегорий

Для каждого непустого фильтра измеряем, какую долю положительных наблюдений покрывают наиболее частые микрокатегории.

In [31]:
filter_microcategory = (
    train_fold
    .filter(filter_present)
    .group_by(["search_infm_params_text", "item_microcat_id"])
    .len()
    .with_columns(
        pl.col("len")
        .rank(method="ordinal", descending=True)
        .over("search_infm_params_text")
        .alias("microcategory_rank")
    )
)

filter_microcategory_coverage = pl.DataFrame([
    {
        "top_k": k,
        "covered_positive_share": filter_microcategory.select(
            pl.col("len")
            .filter(pl.col("microcategory_rank") <= k)
            .sum()
            / pl.col("len").sum()
        ).collect().item(),
    }
    for k in (1, 3, 5)
])

filter_microcategory_coverage

top_k,covered_positive_share
i64,f64
1,0.655403
3,0.842143
5,0.907932


### 9.6. Протокол сравнения retrieval-веток

Каждая ветка должна вернуть одинаковый контракт: `query_norm`, `item_id`, `score` и имя метода. Это позволит честно сравнить отдельные методы и их объединение.

In [32]:
candidate_schema = {
    "query_norm": pl.String,
    "item_id": pl.String,
    "score": pl.Float64,
    "source": pl.String,
}

candidate_results = pl.DataFrame(schema=candidate_schema)
candidate_results

query_norm,item_id,score,source
str,str,f64,str


## 10. Выбор размера пула кандидатов

После реализации хотя бы одной retrieval-ветки измеряем Recall при нескольких размерах пула, не меняя validation-разбиение.

In [38]:
def evaluate_candidate_pool(
    labels: pl.DataFrame,
    candidates: pl.DataFrame,
    cutoffs: tuple[int, ...] = (10, 20, 50, 100, 200, 500),
) -> pl.DataFrame:
    return pl.DataFrame({
        "k": list(cutoffs),
        "recall": [
            recall_at_k(labels, candidates, k=k)
            for k in cutoffs
        ],
    })


candidate_pool_quality = evaluate_candidate_pool(
    validation_labels.collect(),
    candidate_results,
)

## 11. Проверка гипотез ранжирования

К reranker переходим только после того, как генератор кандидатов показывает достаточный Recall на размере пула, доступном по ресурсам.

### 11.1. Признаки ранжирования

Фиксируем группы признаков до обучения, чтобы последующий ablation-анализ был интерпретируемым.

In [39]:
reranking_feature_groups = pl.DataFrame({
    "group": [
        "retrieval",
        "text",
        "geography",
        "category",
        "item_prior",
    ],
    "examples": [
        "score и rank каждой retrieval-ветки",
        "совпадение слов и длины текстов",
        "совпадение локации и вероятность перехода",
        "категория, микрокатегория и фильтры",
        "цена, рейтинг, отзывы и частота в train",
    ],
})

reranking_feature_groups

group,examples
str,str
"""retrieval""","""score и rank каждой retrieval-ветки"""
"""text""","""совпадение слов и длины текстов"""
"""geography""","""совпадение локации и вероятность перехода"""
"""category""","""категория, микрокатегория и фильтры"""
"""item_prior""","""цена, рейтинг, отзывы и частота в train"""


### 11.2. Модель и ablation-анализ

Сначала сравниваем простое объединение retrieval-score с небольшой обучаемой моделью. Затем исключаем по одной группе признаков и измеряем изменение Recall@50 на неизменной validation-выборке.

## 12. Анализ ошибок

После появления первого baseline сохраняем Recall каждого запроса и объединяем его с заранее подготовленными сегментами.

In [40]:
def summarize_recall_by_segment(
    per_query_recall: pl.DataFrame,
    segments: pl.DataFrame,
    segment_column: str,
) -> pl.DataFrame:
    return (
        per_query_recall
        .join(segments, on="query_norm", how="left")
        .group_by(segment_column)
        .agg(
            pl.len().alias("queries"),
            pl.col("recall").mean().alias("mean_recall"),
            pl.col("recall").median().alias("median_recall"),
        )
        .sort("mean_recall")
    )

## 13. Итоговые результаты

Все эксперименты записываются в одну таблицу. Помимо Recall сохраняем размер пула, время работы и короткое описание изменённого компонента.

In [41]:
experiment_columns = {
    "experiment": pl.String,
    "candidate_sources": pl.String,
    "candidate_pool_size": pl.Int64,
    "recall_at_50": pl.Float64,
    "runtime_seconds": pl.Float64,
    "notes": pl.String,
}

experiment_results = pl.DataFrame(schema=experiment_columns)
experiment_results

experiment,candidate_sources,candidate_pool_size,recall_at_50,runtime_seconds,notes
str,str,i64,f64,f64,str


### 13.1. Подтверждённые выводы

В этот раздел попадают только выводы, напрямую подтверждённые таблицами и экспериментами выше.

### 13.2. Ограничения исследования

Здесь фиксируются ограничения proxy-разметки, отличие benchmark от train и возможный разрыв между локальной оценкой и leaderboard.

## 14. Финальный пайплайн

После завершения экспериментов здесь будет зафиксирована итоговая последовательность подготовки данных, генерации кандидатов, ранжирования и формирования `answer.csv`.

## 15. Следующие шаги

Подтверждённая логика переносится из ноутбука в `src/`, после чего создаётся единый воспроизводимый сценарий обучения и инференса.